# SQL Exploration — Part I

Este notebook es el espacio de **prueba y error** (ver `CONVENTIONS.md`). Aquí:
1. Montamos las 4 tablas hipotéticas del enunciado en DuckDB con datos sintéticos.
2. Construimos cada query pieza por pieza, viendo cómo cambia el resultado en cada paso.

La versión final y limpia de cada query (sin los pasos intermedios) se copia después a `sql/sql_queries.sql`.

## 0. Setup de datos sintéticos

Usamos [DuckDB](https://duckdb.org/) porque es un motor SQL que corre embebido en Python (sin instalar un servidor de base de datos aparte), y se lleva muy bien con `pandas`: podemos generar los datos en un DataFrame y luego insertarlos directo en una tabla SQL real.

`np.random.seed(42)` fija la "semilla" del generador de números aleatorios: así, cada vez que corramos este notebook, se generan **los mismos** datos sintéticos (reproducibilidad), en vez de números distintos cada vez.

In [1]:
import duckdb
import numpy as np
import pandas as pd

np.random.seed(42)
con = duckdb.connect(database=":memory:")  # base de datos en memoria, solo para esta sesión

### Las 4 tablas, con el esquema exacto del enunciado

`CREATE TABLE` define el nombre de cada columna y su tipo de dato (`VARCHAR` = texto, `INT` = entero, `FLOAT` = decimal, `TIMESTAMP` = fecha+hora, `DATE` = solo fecha, `BOOLEAN` = verdadero/falso). Todavía no hay ninguna fila de datos, solo la estructura vacía.

**Detalle a tener en cuenta para más adelante:** `deliveries.delivery_person_id` es `VARCHAR` pero `delivery_persons.delivery_person_id` es `INT`. Es una inconsistencia real del enunciado (no un error nuestro) — cuando más adelante hagamos un `JOIN` entre estas dos tablas vamos a necesitar convertir uno de los dos tipos con `CAST` para que coincidan. Por ahora, para la Pregunta 1, no la necesitamos porque no hacemos ningún join.

In [2]:
con.execute("""
CREATE TABLE deliveries (
    delivery_id VARCHAR,
    delivery_person_id VARCHAR,
    restaurant_area VARCHAR,
    customer_area VARCHAR,
    delivery_distance_km FLOAT,
    delivery_time_min INT,
    order_placed_at TIMESTAMP,
    weather_condition VARCHAR,
    traffic_condition VARCHAR,
    delivery_rating FLOAT
)
""")

con.execute("""
CREATE TABLE delivery_persons (
    delivery_person_id INT,
    name VARCHAR,
    region VARCHAR,
    hired_date DATE,
    is_active BOOLEAN
)
""")

con.execute("""
CREATE TABLE restaurants (
    restaurant_id VARCHAR,
    area VARCHAR,
    name VARCHAR,
    cuisine_type VARCHAR,
    avg_preparation_time_min FLOAT
)
""")

con.execute("""
CREATE TABLE orders (
    order_id INT,
    delivery_id VARCHAR,
    restaurant_id VARCHAR,
    customer_id VARCHAR,
    order_value FLOAT,
    items_count INT
)
""")

print("4 tablas creadas (vacías)")

4 tablas creadas (vacías)


### Por qué generamos los datos sintéticos así

Generamos **180 días** de entregas (hoy hacia atrás), con ~14 entregas/día en promedio, distribuidas entre 10 áreas de cliente, 15 repartidores y 20 restaurantes. Usamos `pd.Timestamp.now()` como fecha de referencia (no una fecha fija) para que, sin importar cuándo se vuelva a correr este notebook, "los últimos 30 días" en las queries siempre coincida con datos recientes reales del dataset sintético.

Casos de borde que sembramos a propósito, pensando en las 5 preguntas:

- **Pregunta 1** (top áreas, últimos 30 días): el área **Riverside** tiene un tiempo de entrega normal casi todo el período, pero le agregamos +20 min *solo en los últimos 30 días* (ej. obras viales recientes). El área **Old Town** es al revés: tuvo un problema serio (+25 min) que ya se resolvió, así que solo se nota en datos *anteriores* a los últimos 30 días. Esto sirve para comprobar que el filtro de fecha realmente cambia el ranking — si alguien olvida el `WHERE` de fecha, Old Town aparecería como el peor área, cuando en realidad ya no es un problema actual.
- **Pregunta 3** (repartidores más rápidos, 50+ entregas, activos): el repartidor `3` es rápido, activo y tiene ~500+ entregas. El repartidor `5` es igual de rápido pero **inactivo** — así comprobamos que el filtro `is_active` sí lo excluye (si no filtráramos, aparecería como el más rápido). Los repartidores `14` y `15` son "recién contratados" con muy pocas entregas (<50), para comprobar que el filtro de volumen también los excluye.
- **Pregunta 5** (tendencia creciente): el repartidor `7` tiene un efecto que hace crecer su tiempo de entrega mes a mes (de ~48 a ~60 min), simulando que algo se fue deteriorando (ej. una ruta que empeoró, cansancio, etc.), mientras que el resto de repartidores se mantiene relativamente estable.

Las preguntas 2 y 4 no necesitaron sesgos especiales: con variedad razonable de clima/tráfico/cocina/valores de orden alcanza para responderlas con datos que tengan sentido.

In [3]:
today = pd.Timestamp.now().normalize()
n_days = 180
start_date = today - pd.Timedelta(days=n_days)

areas = ["Downtown", "Uptown", "Riverside", "Midtown", "Eastside",
         "Westside", "Old Town", "Suburbia", "Lakeside", "Hillcrest"]
cuisines = ["Italian", "Fast Food", "Sushi", "Indian", "Mexican", "Vegan", "Pizza", "Chinese"]

# tiempo "base" propio de cada área (ej. áreas más lejanas o más congestionadas)
area_base_time = {a: np.random.uniform(25, 40) for a in areas}

def area_special_effect(area, order_date):
    """Riverside: problema reciente (<=30 días). Old Town: problema ya resuelto (>30 días)."""
    days_ago = (today - order_date).days
    effect = 0.0
    if area == "Riverside" and days_ago <= 30:
        effect += 20.0
    if area == "Old Town" and days_ago > 30:
        effect += 25.0
    return effect

courier_ids = list(range(1, 16))
inactive_ids = {5, 12}       # repartidores que ya no trabajan en la plataforma
new_hire_ids = {14, 15}      # contratados hace poco, todavía con pocas entregas
fast_ids = {3, 5}            # repartidores genuinamente rápidos (uno activo, uno inactivo)
trend_courier = 7            # repartidor con tendencia creciente en tiempo de entrega

weights = []
for cid in courier_ids:
    if cid in fast_ids:
        weights.append(4.0)   # más volumen, para asegurar 50+ entregas
    elif cid in new_hire_ids:
        weights.append(0.05)  # muy poco volumen, para quedar bajo 50 entregas
    else:
        weights.append(1.0)
weights = np.array(weights)
weights /= weights.sum()

print("Setup de repartidores listo")

Setup de repartidores listo


In [4]:
records = []
delivery_counter = 1

for day_offset in range(n_days):
    order_date = start_date + pd.Timedelta(days=day_offset)
    days_ago = (today - order_date).days
    n_today = np.random.poisson(14)

    for _ in range(n_today):
        courier_id = int(np.random.choice(courier_ids, p=weights))
        area = np.random.choice(areas)
        restaurant_area = np.random.choice(areas)
        distance = round(float(np.random.uniform(1, 15)), 2)
        weather = np.random.choice(["Clear", "Rainy", "Cloudy", "Stormy"], p=[0.55, 0.2, 0.2, 0.05])
        traffic = np.random.choice(["Low", "Medium", "High"], p=[0.4, 0.4, 0.2])

        base_time = 20 + distance * 1.5
        weather_effect = {"Clear": 0, "Cloudy": 2, "Rainy": 8, "Stormy": 15}[weather]
        traffic_effect = {"Low": 0, "Medium": 6, "High": 14}[traffic]
        area_effect = area_base_time[area] - 30
        special_effect = area_special_effect(area, order_date)
        courier_effect = -8.0 if courier_id in fast_ids else 0.0
        trend_effect = ((n_days - days_ago) / n_days) * 20 if courier_id == trend_courier else 0.0
        noise = np.random.normal(0, 4)

        delivery_time = (base_time + weather_effect + traffic_effect + area_effect
                          + special_effect + courier_effect + trend_effect + noise)
        delivery_time = int(np.clip(delivery_time, 10, 120))

        order_time = order_date + pd.Timedelta(hours=float(np.random.uniform(8, 22)))
        rating = round(float(np.clip(np.random.normal(4.3, 0.5), 1, 5)), 1)

        records.append({
            "delivery_id": f"D{delivery_counter:06d}",
            "delivery_person_id": str(courier_id),  # VARCHAR, igual que en el esquema de deliveries
            "restaurant_area": restaurant_area,
            "customer_area": area,
            "delivery_distance_km": distance,
            "delivery_time_min": delivery_time,
            "order_placed_at": order_time,
            "weather_condition": weather,
            "traffic_condition": traffic,
            "delivery_rating": rating,
        })
        delivery_counter += 1

deliveries_df = pd.DataFrame(records)
print(f"{len(deliveries_df)} entregas generadas")
deliveries_df.head()

2550 entregas generadas


,delivery_id,delivery_person_id,restaurant_area,customer_area,delivery_distance_km,delivery_time_min,order_placed_at,weather_condition,traffic_condition,delivery_rating
0,D000001,3,Downtown,Eastside,5.26,26,2026-02-07 09:57:10.490576862,Clear,Medium,3.5
1,D000002,3,Old Town,Riverside,14.77,52,2026-02-07 08:11:08.554042457,Clear,High,5.0
2,D000003,12,Hillcrest,Uptown,5.26,39,2026-02-07 14:55:56.916269608,Clear,Medium,4.2
3,D000004,1,Midtown,Downtown,3.55,35,2026-02-07 08:26:18.189939760,Cloudy,Medium,3.6
4,D000005,11,Hillcrest,Uptown,13.53,71,2026-02-07 13:26:29.335400349,Rainy,High,4.1


In [5]:
# delivery_persons
regions = ["North", "South", "East", "West"]
dp_records = []
for cid in courier_ids:
    if cid in new_hire_ids:
        hired = today - pd.Timedelta(days=int(np.random.uniform(5, 20)))
    else:
        hired = today - pd.Timedelta(days=int(np.random.uniform(60, 1000)))
    dp_records.append({
        "delivery_person_id": cid,
        "name": f"Courier_{cid:02d}",
        "region": np.random.choice(regions),
        "hired_date": hired.date(),
        "is_active": cid not in inactive_ids,
    })
delivery_persons_df = pd.DataFrame(dp_records)

# restaurants: 2 por área, para tener variedad de área/cocina
r_records = []
rid = 1
for area in areas:
    for _ in range(2):
        r_records.append({
            "restaurant_id": f"R{rid:03d}",
            "area": area,
            "name": f"Restaurant_{rid:03d}",
            "cuisine_type": np.random.choice(cuisines),
            "avg_preparation_time_min": round(float(np.random.uniform(10, 30)), 1),
        })
        rid += 1
restaurants_df = pd.DataFrame(r_records)
restaurants_by_area = restaurants_df.groupby("area")["restaurant_id"].apply(list).to_dict()

# orders: una orden por entrega, con un restaurante de la misma área que restaurant_area
o_records = []
for i, row in deliveries_df.iterrows():
    restaurant_id = np.random.choice(restaurants_by_area[row["restaurant_area"]])
    items = int(np.random.randint(1, 8))
    order_value = round(items * float(np.random.uniform(8, 20)), 2)
    o_records.append({
        "order_id": i + 1,
        "delivery_id": row["delivery_id"],
        "restaurant_id": restaurant_id,
        "customer_id": f"C{np.random.randint(1, 500):04d}",
        "order_value": order_value,
        "items_count": items,
    })
orders_df = pd.DataFrame(o_records)

print("delivery_persons, restaurants y orders generados")

delivery_persons, restaurants y orders generados


### Insertar los DataFrames en las tablas SQL

`INSERT INTO tabla SELECT * FROM df` copia las filas del DataFrame de pandas a la tabla de DuckDB. DuckDB reconoce automáticamente las variables de Python que tienen un DataFrame (por eso podemos escribir `FROM deliveries_df` directamente dentro del SQL, sin pasos extra de registro).

In [6]:
con.execute("INSERT INTO deliveries SELECT * FROM deliveries_df")
con.execute("INSERT INTO delivery_persons SELECT * FROM delivery_persons_df")
con.execute("INSERT INTO restaurants SELECT * FROM restaurants_df")
con.execute("INSERT INTO orders SELECT * FROM orders_df")

for t in ["deliveries", "delivery_persons", "restaurants", "orders"]:
    n = con.execute(f"SELECT COUNT(*) FROM {t}").fetchone()[0]
    print(f"{t}: {n} filas")

deliveries: 2550 filas
delivery_persons: 15 filas
restaurants: 20 filas
orders: 2550 filas


### Verificación rápida de los casos de borde

Antes de pasar a las queries, confirmamos que los datos sintéticos efectivamente tienen las propiedades que buscábamos.

In [7]:
# Riverside (problema reciente) vs Old Town (problema ya resuelto)
con.sql("""
    SELECT
        customer_area,
        ROUND(AVG(delivery_time_min), 1) AS avg_all_time,
        ROUND(AVG(CASE WHEN order_placed_at >= CURRENT_DATE - INTERVAL 30 DAY
                       THEN delivery_time_min END), 1) AS avg_last_30d
    FROM deliveries
    WHERE customer_area IN ('Riverside', 'Old Town')
    GROUP BY customer_area
""").df()

,customer_area,avg_all_time,avg_last_30d
0,Riverside,45.7,62.2
1,Old Town,55.0,34.2


In [8]:
# Repartidores clave para la Pregunta 3: volumen y estado activo/inactivo
con.sql("""
    SELECT
        dp.delivery_person_id,
        dp.is_active,
        COUNT(*) AS n_deliveries,
        ROUND(AVG(d.delivery_time_min), 1) AS avg_time
    FROM deliveries d
    JOIN delivery_persons dp ON d.delivery_person_id = CAST(dp.delivery_person_id AS VARCHAR)
    WHERE dp.delivery_person_id IN (3, 5, 14, 15)
    GROUP BY dp.delivery_person_id, dp.is_active
    ORDER BY dp.delivery_person_id
""").df()

,delivery_person_id,is_active,n_deliveries,avg_time
0,3,True,537,37.1
1,5,False,539,37.6
2,14,True,6,38.0
3,15,True,9,42.7


In [9]:
# Repartidor 7: tendencia creciente mes a mes (para la Pregunta 5)
con.sql("""
    SELECT
        DATE_TRUNC('month', order_placed_at) AS month,
        ROUND(AVG(delivery_time_min), 1) AS avg_time,
        COUNT(*) AS n
    FROM deliveries
    WHERE delivery_person_id = '7'
    GROUP BY 1
    ORDER BY 1
""").df()

,month,avg_time,n
0,2026-02-01,48.0,17
1,2026-03-01,52.0,15
2,2026-04-01,56.2,21
3,2026-05-01,60.3,18
4,2026-06-01,59.2,20
5,2026-07-01,57.5,21
6,2026-08-01,60.3,7


Confirmado:
- **Old Town** tiene el promedio histórico más alto (~55 min) pero baja mucho en los últimos 30 días (~34 min) — el problema ya se resolvió. **Riverside** es lo opuesto: promedio histórico moderado (~46 min) pero se dispara en los últimos 30 días (~62 min) — problema reciente. Si olvidáramos el filtro de fecha en la Pregunta 1, Old Town aparecería como el peor, escondiendo el problema real y actual de Riverside.
- El repartidor `3` (activo) y el `5` (inactivo) tienen ambos 500+ entregas y tiempos rápidos similares — sin el filtro `is_active`, el `5` aparecería en el ranking de más rápidos aunque ya no trabaje ahí. Los repartidores `14` y `15` tienen menos de 50 entregas, así que quedarán fuera del ranking por volumen insuficiente.
- El repartidor `7` pasa de ~48 min en promedio a ~60 min a lo largo de los 6 meses — una tendencia creciente clara, aunque con algo de ruido mes a mes (como pasaría con datos reales).

Con los datos ya listos, pasamos a construir las queries.

## Pregunta 1: Top 5 áreas de cliente con mayor tiempo promedio de entrega en los últimos 30 días

La vamos a construir en 3 pasos: primero el `GROUP BY` + `AVG()` básico (sobre todos los datos), luego agregamos el filtro de fecha, y por último el `ORDER BY` + `LIMIT` para quedarnos con el top 5.

### Paso 1: `GROUP BY` + `AVG()` básico

`GROUP BY customer_area` agrupa todas las filas de `deliveries` que tienen la misma área de cliente en un solo "bloque" por área. `AVG(delivery_time_min)` calcula, dentro de cada bloque, el promedio de la columna `delivery_time_min`. El resultado es una fila por área, con su tiempo promedio de entrega — pero **usando todo el historial**, todavía sin filtrar por fecha, y en un orden arbitrario (todavía no hay `ORDER BY`).

In [10]:
con.sql("""
    SELECT
        customer_area,
        AVG(delivery_time_min) AS avg_delivery_time_min
    FROM deliveries
    GROUP BY customer_area
""").df()

,customer_area,avg_delivery_time_min
0,Riverside,45.688000
1,Uptown,45.551724
2,Downtown,37.324219
3,Midtown,39.848249
4,Westside,32.966292
5,Hillcrest,43.175781
6,Lakeside,41.877049
7,Eastside,33.558140
8,Old Town,55.018519
9,Suburbia,45.017316


Vemos las 10 áreas con su promedio histórico. Nota que **Old Town** aparece con el promedio más alto (~55 min) — pero, como vimos en la verificación de arriba, ese problema ya se resolvió y no debería contar para "los últimos 30 días". Por eso necesitamos el siguiente paso.

### Paso 2: agregar el filtro de fecha (últimos 30 días)

`WHERE order_placed_at >= CURRENT_DATE - INTERVAL 30 DAY` filtra las filas *antes* de agruparlas: solo entran al cálculo las entregas cuya fecha/hora de pedido (`order_placed_at`) sea de los últimos 30 días. `CURRENT_DATE` es la fecha de hoy (según el reloj de la máquina donde corre la query), y `- INTERVAL 30 DAY` le resta 30 días. El orden importa: SQL aplica el `WHERE` antes que el `GROUP BY`, así que el promedio ya se calcula solo con las filas recientes.

In [11]:
con.sql("""
    SELECT
        customer_area,
        AVG(delivery_time_min) AS avg_delivery_time_min
    FROM deliveries
    WHERE order_placed_at >= CURRENT_DATE - INTERVAL 30 DAY
    GROUP BY customer_area
""").df()

,customer_area,avg_delivery_time_min
0,Eastside,32.675000
1,Downtown,38.275000
2,Westside,33.022727
3,Midtown,41.535714
4,Old Town,34.153846
5,Lakeside,42.022222
6,Hillcrest,42.222222
7,Riverside,62.238095
8,Uptown,47.055556
9,Suburbia,47.307692


El resultado cambió como esperábamos: **Old Town** ahora tiene uno de los promedios más bajos (~34 min), y **Riverside** subió a ser el más alto (~62 min). Este es exactamente el punto del ejercicio: sin el filtro de fecha, habríamos señalado como "problema" un área que ya se recuperó, y nos habríamos perdido el problema real y actual.

### Paso 3: `ORDER BY` + `LIMIT` (top 5)

`ORDER BY avg_delivery_time_min DESC` ordena las filas de mayor a menor promedio (`DESC` = descendente; queremos las de *mayor* tiempo primero, ya que buscamos las áreas con más demora). `LIMIT 5` corta el resultado y deja solo las primeras 5 filas — el top 5 que pide la pregunta.

In [12]:
con.sql("""
    SELECT
        customer_area,
        AVG(delivery_time_min) AS avg_delivery_time_min
    FROM deliveries
    WHERE order_placed_at >= CURRENT_DATE - INTERVAL 30 DAY
    GROUP BY customer_area
    ORDER BY avg_delivery_time_min DESC
    LIMIT 5
""").df()

,customer_area,avg_delivery_time_min
0,Riverside,62.238095
1,Suburbia,47.307692
2,Uptown,47.055556
3,Hillcrest,42.222222
4,Lakeside,42.022222


Esta es la query completa para la Pregunta 1: **Riverside, Suburbia, Uptown, Hillcrest y Lakeside** son las 5 áreas con mayor tiempo promedio de entrega en los últimos 30 días, con Riverside claramente por encima del resto (~62 min vs ~42-47 min de las siguientes).

**Nos detenemos aquí** — según lo acordado, revisamos esta query juntos antes de pasarla a `sql_queries.sql` y antes de seguir con las preguntas 2-5.